# Senior Scam Shield: ML risk score on IBM Telum

The rules engine decides the action (allow / warn / pause / hold / escalate). This notebook trains a
small model that adds a **scam risk score** to each decision. **The rules keep deciding; the model only
adds a number.**

What this notebook does, top to bottom:
1. Checks the machine: `s390x`, the Telum `nnpa` flag, library versions, free memory.
2. Generates a synthetic population (6,000 customers, ~2% scam victims) and runs every event through
   the real engine to get feature rows.
3. **Honest test:** trains on two scam types and tests on the third, which the model has never seen.
   Two frameworks: scikit-learn (gradient boosting) and TensorFlow/Keras (a small neural network).
4. Trains the final network and exports it to `ml/model.json`, which `ml/scorer.py` runs in plain Python.
5. Times inference on a large batch (the Telum part).
6. Checks the engine tests still pass and scores Shums's 8 real scenarios.

All data is synthetic. We wrote both the scam patterns and the rules, so a model tested on scam types it
was trained on would score ~99% and prove nothing. That's why step 3 holds one scam type out.

In [ ]:
import os, sys, platform, subprocess, time, json, collections, contextlib
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# Find the repo root (this notebook lives in ml/).
ROOT = Path.cwd()
while not (ROOT / "engine" / "engine.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "engine" / "engine.py").exists(), "Open this notebook from inside the repo's ml/ folder"
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import sklearn
import tensorflow as tf

def nnpa_status():
    try:
        out = subprocess.run(["lscpu"], capture_output=True, text=True).stdout.lower()
        if out:
            return "PRESENT (lscpu flags include nnpa)" if " nnpa" in out else "not found in lscpu"
    except FileNotFoundError:
        pass
    try:
        return "PRESENT (/proc/cpuinfo features include nnpa)" if "nnpa" in Path("/proc/cpuinfo").read_text().lower() else "not found"
    except OSError:
        return "unknown (no lscpu or /proc/cpuinfo)"

def mem_available_gb():
    try:
        for line in Path("/proc/meminfo").read_text().splitlines():
            if line.startswith("MemAvailable"):
                return f"{int(line.split()[1]) / 1e6:.1f} GB"
    except OSError:
        return "unknown"

print("machine         :", platform.machine())
print("Telum nnpa flag :", nnpa_status())
print("python          :", platform.python_version())
print("numpy           :", np.__version__)
print("scikit-learn    :", sklearn.__version__)
print("tensorflow      :", tf.__version__)
print("TF devices      :", [f"{d.device_type}:{d.name}" for d in tf.config.list_physical_devices()])
print("memory available:", mem_available_gb())

## 1. Generate a population and build feature rows

Every customer gets ~90 days of normal history (Shums's `data/make_history.py`, rescaled per customer)
and a normal fortnight of live activity. About 12% also get an innocent look-alike (house purchase, tax
bill, car bought with cash, contractor, gift to family), and about 2% get a randomised scam chain.

Each live event goes through the real engine. The features come from `Engine.features()` via the scorer
hook (plus payee trust, see `ml/features.py`), so the model sees exactly what the rules see. Only this
notebook reads the `truth` label; the engine never does.

In [ ]:
from ml.population import iter_population, build_dataset, SCAM_TYPES, LOOKALIKE_TYPES
from ml.features import to_vector, RULE_FEATURES

N_CUSTOMERS = 6000
t0 = time.perf_counter()
rows = build_dataset(iter_population(N_CUSTOMERS, seed=7))
print(f"{N_CUSTOMERS} customers -> {len(rows)} live events in {time.perf_counter() - t0:.1f}s")

kinds_per_customer = collections.Counter({r["customer_id"]: r["kind"] for r in rows}.values())
print("customers by kind:", dict(kinds_per_customer.most_common()))
truth = collections.Counter(r["truth"] for r in rows)
print(f"scam events: {truth['scam']} of {len(rows)} ({truth['scam'] / len(rows):.2%}), "
      f"scam victims: {sum(kinds_per_customer[k] for k in SCAM_TYPES) / N_CUSTOMERS:.1%} of customers")

In [ ]:
NAMES = list(rows[0]["features"])
X = np.array([to_vector(r["features"], NAMES) for r in rows], dtype="float32")
y = np.array([r["truth"] == "scam" for r in rows], dtype="float32")
kind = np.array([r["kind"] for r in rows])
cid = np.array([r["customer_id"] for r in rows])
outflow = np.array([r["features"]["is_outflow"] == 1 for r in rows])          # money leaving the bank
rules_stop = np.array([r["action"] in ("pause", "hold", "escalate") for r in rows])
test_customer = np.array([int(c[1:]) % 10 < 3 for c in cid])                   # 30% of customers, fixed by id
print(f"{X.shape[0]} rows x {X.shape[1]} features: {NAMES}")

## 2. The honest test: train on two scam types, test on the unseen third

**What counts.** A model "catches" a scam victim if it flags one of their scam events that moves money
out of the bank (payment, wire, cash). Flagging a transfer between someone's own accounts would stop
nothing, and at that first step a home buyer and a gold bar victim look the same (the rules only *warn*
there too).

**Same false-alarm budget as the rules.** The threshold is set on the training customers only, so the
model wrongly flags no more innocent customers than the rules do. Then we see how many victims of the
unseen scam type it still catches.

Rows: scikit-learn gradient boosting, the same without the two features built from the rules' output
(`chain_length`, `triggers_fired`), and the Keras network.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import average_precision_score

tf.keras.utils.set_random_seed(0)

def scam_weight(y):
    return float((len(y) - y.sum()) / max(y.sum(), 1))

def train_gb(X, y):
    w = np.where(y == 1, scam_weight(y), 1.0)
    model = GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=0).fit(X, y, sample_weight=w)
    return model, (lambda Z: model.predict_proba(Z)[:, 1])

def build_mlp(n_features):
    return tf.keras.Sequential([
        tf.keras.Input(shape=(n_features,)),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(8, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])

def train_mlp(X, y, epochs=15):
    mu, sd = X.mean(axis=0), X.std(axis=0) + 1e-6
    model = build_mlp(X.shape[1])
    model.compile(optimizer="adam", loss="binary_crossentropy")
    model.fit((X - mu) / sd, y, epochs=epochs, batch_size=256, verbose=0,
              class_weight={0: 1.0, 1: scam_weight(y)})
    return (model, mu, sd), (lambda Z: model.predict((Z - mu) / sd, batch_size=4096, verbose=0).ravel())

def budget_threshold(scores, y, cid, outflow, rules_stop):
    """Lowest threshold that flags no more innocent customers than the rules flag (training data only)."""
    innocent = set(cid[y == 0]) - set(cid[y == 1])
    budget = len(set(cid[rules_stop & (y == 0)]) & innocent)
    top = {}
    for c, s, o in zip(cid, scores, outflow):
        if o and c in innocent:
            top[c] = max(top.get(c, 0.0), float(s))
    ranked = sorted(top.values(), reverse=True) + [0.0] * (len(innocent) - len(top))
    return ranked[budget] + 1e-9, budget, len(innocent)

def report(name, held, scores, thr, te):
    flag = (scores >= thr) & outflow[te]
    k, c, yy = kind[te], cid[te], y[te]
    victims = set(c[k == held])
    caught = set(c[(k == held) & flag & (yy == 1)])
    innocent = set(c[yy == 0]) - victims
    false_alarms = set(c[flag & (yy == 0)]) & innocent
    lookalikes = set(c[np.isin(k, LOOKALIKE_TYPES)])
    return {"held_out": held, "model": name, "caught": len(caught), "victims": len(victims),
            "false_alarms": len(false_alarms), "innocent": len(innocent),
            "lookalike_fa": len(false_alarms & lookalikes), "lookalikes": len(lookalikes),
            "avg_precision": average_precision_score(yy, scores)}

rule_cols = [i for i, n in enumerate(NAMES) if n not in RULE_FEATURES]
results = []
t0 = time.perf_counter()
for held in SCAM_TYPES:
    others = [s for s in SCAM_TYPES if s != held]
    tr = ~test_customer & (kind != held)                         # never sees the held-out type
    te = (test_customer & ~np.isin(kind, others)) | (kind == held)
    for name, trainer, cols in [("scikit-learn GB", train_gb, slice(None)),
                                ("scikit-learn GB, no rule features", train_gb, rule_cols),
                                ("Keras MLP", train_mlp, slice(None))]:
        _, predict = trainer(X[tr][:, cols], y[tr])
        thr, budget, n_innocent = budget_threshold(predict(X[tr][:, cols]), y[tr], cid[tr], outflow[tr], rules_stop[tr])
        results.append(report(name, held, predict(X[te][:, cols]), thr, te))
    # The rules, scored the same way, for comparison.
    flag = rules_stop[te]
    c, k, yy = cid[te], kind[te], y[te]
    victims = set(c[k == held]); innocent = set(c[yy == 0]) - victims
    results.append({"held_out": held, "model": "rules engine (has a rule for it)",
                    "caught": len(set(c[(k == held) & flag & (yy == 1)])), "victims": len(victims),
                    "false_alarms": len(set(c[flag & (yy == 0)]) & innocent), "innocent": len(innocent),
                    "lookalike_fa": len(set(c[flag & (yy == 0)]) & set(c[np.isin(k, LOOKALIKE_TYPES)])),
                    "lookalikes": len(set(c[np.isin(k, LOOKALIKE_TYPES)])), "avg_precision": float("nan")})
print(f"trained {3 * 3} models in {time.perf_counter() - t0:.0f}s. False-alarm budget: {budget} of {n_innocent} innocent training customers\n")

print(f"{'held-out scam type':18} {'model':36} {'caught':>12} {'false alarms':>16} {'look-alike FA':>15} {'avg prec':>9}")
for r in results:
    print(f"{r['held_out']:18} {r['model']:36} {r['caught']:>4}/{r['victims']:<3} {r['caught'] / r['victims']:>4.0%} "
          f"{r['false_alarms']:>5}/{r['innocent']:<5} {r['false_alarms'] / r['innocent']:>5.1%} "
          f"{r['lookalike_fa']:>5}/{r['lookalikes']:<4} {r['avg_precision']:>9.2f}")

**How to read this.** "caught" = victims of a scam type the model was **never trained on** who got at
least one money-leaving scam event flagged. "false alarms" = innocent test customers flagged at least
once. The rules row is not a fair fight: the rules were written for all three scams. It is there to show
the false-alarm level the model is held to.

In [ ]:
gb_all, _ = train_gb(X, y)
order = np.argsort(gb_all.feature_importances_)[::-1]
print("Which features the gradient boosting model relies on (trained on all data):\n")
for i in order:
    print(f"{NAMES[i]:32} {gb_all.feature_importances_[i]:6.3f} " + "#" * int(60 * gb_all.feature_importances_[i]))

## 3. Train the final network and export it

Trained on all three scam types. The weights go to `ml/model.json`; `ml/scorer.py` runs the same maths in
plain Python, so the engine and the API container can use it with nothing to install. The check below
compares the plain-Python scores with Keras on 1,000 rows.

In [ ]:
(final_model, MU, SD), final_predict = train_mlp(X, y)
final_thr, budget, n_innocent = budget_threshold(final_predict(X), y, cid, outflow, rules_stop)

export = {
    "name": "keras-mlp",
    "features": NAMES,
    "mean": MU.tolist(),
    "std": SD.tolist(),
    "threshold": float(final_thr),
    "layers": [
        {"W": layer.get_weights()[0].tolist(), "b": layer.get_weights()[1].tolist(),
         "activation": layer.get_config()["activation"]}
        for layer in final_model.layers
    ],
    "trained_on": f"{N_CUSTOMERS} synthetic customers, {len(rows)} events, tensorflow {tf.__version__}, {platform.machine()}",
}
model_path = ROOT / "ml" / "model.json"
model_path.write_text(json.dumps(export))
print(f"wrote {model_path.relative_to(ROOT)} ({model_path.stat().st_size / 1024:.0f} KB), threshold {final_thr:.3f}")

from ml.scorer import load_model, predict as plain_predict
plain = load_model(model_path)
sample = np.random.default_rng(0).choice(len(rows), 1000, replace=False)
diff = max(abs(plain_predict(plain, rows[i]["features"]) - p) for i, p in zip(sample, final_predict(X[sample])))
print(f"plain-Python scorer vs Keras, max difference over 1,000 rows: {diff:.2e}")

## 4. Inference speed on LinuxONE (Telum)

Telum's on-chip AI accelerator (the `nnpa` CPU flag) speeds up neural-network maths, so this times the
Keras network on a large batch. If TensorFlow exposes an NNPA device, the same batch is also timed with
the work pinned to the CPU, for a before/after. **These are the numbers we measured, nothing more:** if no
NNPA device shows up, the time below is plain CPU and we say so.

In [ ]:
N_BIG = 200_000
Z = ((np.tile(X, (N_BIG // len(X) + 1, 1))[:N_BIG] - MU) / SD).astype("float32")

def time_keras(device=None, batch=8192, repeats=3):
    ctx = tf.device(device) if device else contextlib.nullcontext()
    with ctx:
        final_model(tf.constant(Z[:batch]), training=False)              # warm-up
        best = float("inf")
        for _ in range(repeats):
            t = time.perf_counter()
            for i in range(0, len(Z), batch):
                final_model(tf.constant(Z[i:i + batch]), training=False).numpy()
            best = min(best, time.perf_counter() - t)
    return len(Z) / best

devices = tf.config.list_physical_devices()
nnpa = [d for d in devices if "NNPA" in d.device_type.upper()]
print("machine:", platform.machine(), "| nnpa flag:", nnpa_status())
print("TensorFlow devices:", [f"{d.device_type}:{d.name}" for d in devices])

timing = {"default": time_keras()}
print(f"\nKeras network, default placement: {timing['default']:>12,.0f} events/second  ({N_BIG:,} events)")
if nnpa:
    timing["cpu"] = time_keras("/CPU:0")
    print(f"Keras network, pinned to CPU:     {timing['cpu']:>12,.0f} events/second")
    print(f"speed-up from NNPA placement:     {timing['default'] / timing['cpu']:.2f}x")
else:
    print("No NNPA device is registered in this TensorFlow, so the number above is CPU time.")

t = time.perf_counter(); gb_all.predict_proba(Z[:50_000] * SD + MU); gb_rate = 50_000 / (time.perf_counter() - t)
print(f"scikit-learn GB (CPU, for comparison): {gb_rate:>8,.0f} events/second")

from ml.scorer import load_scorer
score = load_scorer(model_path)
t = time.perf_counter()
for r in rows[:2000]:
    score(r["features"])
print(f"plain-Python scorer, one event at a time: {(time.perf_counter() - t) / 2000 * 1000:.3f} ms per event")

## 5. Nothing broke: engine tests, and the 8 real scenarios with the score attached

`MLEngine` only adds features; actions must be identical to the plain engine.

In [ ]:
test = subprocess.run([sys.executable, "-m", "unittest", "engine.test_engine"], capture_output=True, text=True, cwd=ROOT)
print("engine tests:", test.stderr.strip().splitlines()[-1])

from engine.engine import Engine
from ml.features import MLEngine

scenarios = [json.loads(p.read_text()) for p in sorted((ROOT / "data" / "scenarios").glob("*.json")) if not p.name.startswith("_")]
plain_engine, ml_engine = Engine(), MLEngine(scorer=score)
for s in scenarios:
    plain_engine.register(s["customer"], s["history"])
    ml_engine.register(s["customer"], s["history"])
live = sorted((e for s in scenarios for e in s["live"]), key=lambda e: e["timestamp"])
names = {s["customer"]["customer_id"]: s["scenario"] for s in scenarios}
same = True
print(f"\n{'scenario':16} {'event':18} {'amount':>10} {'truth':7} {'action':9} {'ML risk':>7}  flag")
for e in live:
    d_plain, d_ml = plain_engine.process(e), ml_engine.process(e)
    same &= d_plain["action"] == d_ml["action"]
    print(f"{names[e['customer_id']]:16} {e['type']:18} {e['amount']:>10,.0f} {e['truth']:7} {d_ml['action']:9} "
          f"{d_ml['ml']['risk']:>7.3f}  {'FLAG' if d_ml['ml']['flag'] else ''}")
print("\nactions identical with and without the model:", same)

## 6. The sentences for the pitch

In [ ]:
mlp = [r for r in results if r["model"] == "Keras MLP"]
gb = [r for r in results if r["model"] == "scikit-learn GB"]
line = lambda rs: ", ".join(f"{r['held_out'].replace('_', ' ')} {r['caught']}/{r['victims']}" for r in rs)
fa = lambda rs: f"{min(r['false_alarms'] / r['innocent'] for r in rs):.1%} to {max(r['false_alarms'] / r['innocent'] for r in rs):.1%}"
print(f"""On top of the rules, a small neural network (TensorFlow/Keras) and a gradient-boosting model
(scikit-learn) score how scam-like each customer's last 7 days look. Tested on a scam type it was never
trained on, the network still caught {line(mlp)} victims (gradient boosting: {line(gb)}),
with {fa(mlp)} of innocent customers flagged, the same false-alarm budget as our rules.
On IBM LinuxONE ({platform.machine()}, Telum nnpa: {nnpa_status().split()[0].lower()}) it scores {timing['default']:,.0f} events per second.""")